In [1]:
import pandas as pd
import numpy as np
import warnings
from pandas.errors import PerformanceWarning
warnings.filterwarnings("ignore", category=PerformanceWarning)
pd.set_option('future.no_silent_downcasting', True)

# **1. UCDP**
---

The UCDP Georeferenced Event Dataset (GED) is processed to construct a country–month panel of conflict intensity.

1. **Sample restriction:** The dataset is restricted to state-based conflicts (type 1 violence), which are the only conflicts in which peace agreements are signed.

2. **Country harmonization:** Country names are mapped to ISO3 codes using an external concordance table, and regional classifications are completed to ensure consistency.

3. **Monthly expansion:** Conflict spells defined by start and end dates are expanded into monthly observations.

4. **Fatalities allocation:** Total fatalities are evenly distributed across the months spanned by each conflict spell.

5. **Aggregation:** Monthly observations are aggregated to the country–month level, yielding measures of conflict intensity, number of conflicts, and number of events.


In [2]:
df_ucdp = pd.read_csv('../../data/input/ucdp/GEDEvent_v25_1.csv', low_memory=False)
#filter by type 1 of conflict because are the only ones that sign agreements
df_ucdp = df_ucdp[df_ucdp['type_of_violence']==1].reset_index(drop=True).copy()
relevant_columns = ['country','conflict_new_id', 'date_start','date_end','best', 'dyad_new_id', 'type_of_violence', 'region','conflict_name', 'side_a', 'side_b']
df_ucdp = df_ucdp[relevant_columns].sort_values(['country', 'conflict_new_id', 'date_start'])

#join with isocode and add isocode
isocodes = pd.read_csv('../../data/input/isocodes/isocodes_appended.csv', sep = ';')
#fill missing region and sub_region groupingby alpha_3 ffill and bfill
isocodes[['region', 'sub_region']] = (
    isocodes
    .groupby('alpha_3')[['region', 'sub_region']]
    .transform(lambda s: s.ffill().bfill())    
)
df_ucdp = df_ucdp.merge(isocodes[['name', 'alpha_3', 'sub_region']], left_on='country', right_on='name', how='left').rename(columns = {'alpha_3':'isocode'})
df_ucdp

,country,conflict_new_id,date_start,date_end,best,dyad_new_id,type_of_violence,region,conflict_name,side_a,side_b,name,isocode,sub_region
0,Afghanistan,259,2017-07-31 00:00:00.000,2017-07-31 00:00:00.000,6,524,1,Asia,Iraq: Government,Government of Iraq,IS,Afghanistan,AFG,Southern Asia
1,Afghanistan,259,2021-08-26 00:00:00.000,2021-08-26 00:00:00.000,183,524,1,Asia,Iraq: Government,Government of Iraq,IS,Afghanistan,AFG,Southern Asia
2,Afghanistan,259,2021-08-28 00:00:00.000,2021-08-28 00:00:00.000,2,524,1,Asia,Iraq: Government,Government of Iraq,IS,Afghanistan,AFG,Southern Asia
3,Afghanistan,259,2021-08-29 00:00:00.000,2021-08-29 00:00:00.000,10,524,1,Asia,Iraq: Government,Government of Iraq,IS,Afghanistan,AFG,Southern Asia
4,Afghanistan,333,1989-01-01 00:00:00.000,1989-01-01 00:00:00.000,4,727,1,Asia,Afghanistan: Government,Government of Afghanistan,Hizb-i Islami-yi Afghanistan - Khalis faction,Afghanistan,AFG,Southern Asia
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
271326,Yemen (North Yemen),16099,2024-05-30 00:00:00.000,2024-05-30 00:00:00.000,14,17738,1,Middle East,"United Kingdom, United States of America - Yemen","Government of United Kingdom, Government of Un...",Government of Yemen (North Yemen),Yemen (North Yemen),YEM,Western Asia
271327,Yemen (North Yemen),16099,2024-06-13 00:00:00.000,2024-06-13 00:00:00.000,5,17738,1,Middle East,"United Kingdom, United States of America - Yemen","Government of United Kingdom, Government of Un...",Government of Yemen (North Yemen),Yemen (North Yemen),YEM,Western Asia
271328,Yemen (North Yemen),16099,2024-09-10 00:00:00.000,2024-09-10 00:00:00.000,2,17738,1,Middle East,"United Kingdom, United States of America - Yemen","Government of United Kingdom, Government of Un...",Government of Yemen (North Yemen),Yemen (North Yemen),YEM,Western Asia
271329,Yemen (North Yemen),16099,2024-11-12 00:00:00.000,2024-11-12 00:00:00.000,10,17738,1,Middle East,"United Kingdom, United States of America - Yemen","Government of United Kingdom, Government of Un...",Government of Yemen (North Yemen),Yemen (North Yemen),YEM,Western Asia


In [3]:
df_ucdp["date_start"] = pd.to_datetime(df_ucdp["date_start"]).dt.to_period("M").dt.to_timestamp()
df_ucdp["date_end"]   = pd.to_datetime(df_ucdp["date_end"]).dt.to_period("M").dt.to_timestamp()

# Number of months in each row (inclusive)
n_months = (
    (df_ucdp["date_end"].dt.to_period("M") - df_ucdp["date_start"].dt.to_period("M"))
    .apply(lambda x: x.n + 1)
)

# Repeat rows for each month
df_expanded = df_ucdp.loc[df_ucdp.index.repeat(n_months)].copy()

month_offsets = df_expanded.groupby(level=0).cumcount()

df_expanded["date"] = (
    df_expanded["date_start"].dt.to_period("M")
    .add(month_offsets.values)
    .dt.to_timestamp()
)

df_expanded["best"] = df_expanded["best"] / n_months.repeat(n_months).values
df_expanded

,country,conflict_new_id,date_start,date_end,best,dyad_new_id,type_of_violence,region,conflict_name,side_a,side_b,name,isocode,sub_region,date
0,Afghanistan,259,2017-07-01,2017-07-01,6.0,524,1,Asia,Iraq: Government,Government of Iraq,IS,Afghanistan,AFG,Southern Asia,2017-07-01
1,Afghanistan,259,2021-08-01,2021-08-01,183.0,524,1,Asia,Iraq: Government,Government of Iraq,IS,Afghanistan,AFG,Southern Asia,2021-08-01
2,Afghanistan,259,2021-08-01,2021-08-01,2.0,524,1,Asia,Iraq: Government,Government of Iraq,IS,Afghanistan,AFG,Southern Asia,2021-08-01
3,Afghanistan,259,2021-08-01,2021-08-01,10.0,524,1,Asia,Iraq: Government,Government of Iraq,IS,Afghanistan,AFG,Southern Asia,2021-08-01
4,Afghanistan,333,1989-01-01,1989-01-01,4.0,727,1,Asia,Afghanistan: Government,Government of Afghanistan,Hizb-i Islami-yi Afghanistan - Khalis faction,Afghanistan,AFG,Southern Asia,1989-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
271326,Yemen (North Yemen),16099,2024-05-01,2024-05-01,14.0,17738,1,Middle East,"United Kingdom, United States of America - Yemen","Government of United Kingdom, Government of Un...",Government of Yemen (North Yemen),Yemen (North Yemen),YEM,Western Asia,2024-05-01
271327,Yemen (North Yemen),16099,2024-06-01,2024-06-01,5.0,17738,1,Middle East,"United Kingdom, United States of America - Yemen","Government of United Kingdom, Government of Un...",Government of Yemen (North Yemen),Yemen (North Yemen),YEM,Western Asia,2024-06-01
271328,Yemen (North Yemen),16099,2024-09-01,2024-09-01,2.0,17738,1,Middle East,"United Kingdom, United States of America - Yemen","Government of United Kingdom, Government of Un...",Government of Yemen (North Yemen),Yemen (North Yemen),YEM,Western Asia,2024-09-01
271329,Yemen (North Yemen),16099,2024-11-01,2024-11-01,10.0,17738,1,Middle East,"United Kingdom, United States of America - Yemen","Government of United Kingdom, Government of Un...",Government of Yemen (North Yemen),Yemen (North Yemen),YEM,Western Asia,2024-11-01


In [4]:
#2. Aggregate to country-month level and sum the 'best' values without distinguishing countries
#-----------------------------------------------------------------------------------------------
df_country_month = (
    df_expanded
    .groupby(['isocode','country','date'], as_index=False)
    .agg(
        best=('best','sum'),
        n_conflicts=('conflict_new_id', 'nunique'),
        n_events=('dyad_new_id', 'count'),
        region = ('region','first'),
        subregion = ('sub_region','first')
    )
)
df_country_month = df_country_month.rename(columns={'date':'year_mo'})
df_country_month

,isocode,country,year_mo,best,n_conflicts,n_events,region,subregion
0,AFG,Afghanistan,1989-01-01,693.750000,1,13,Asia,Southern Asia
1,AFG,Afghanistan,1989-02-01,162.750000,1,18,Asia,Southern Asia
2,AFG,Afghanistan,1989-03-01,1745.750000,1,21,Asia,Southern Asia
3,AFG,Afghanistan,1989-04-01,495.750000,1,28,Asia,Southern Asia
4,AFG,Afghanistan,1989-05-01,441.000000,1,18,Asia,Southern Asia
...,...,...,...,...,...,...,...,...
11597,YEM,Yemen (North Yemen),2024-11-01,26.166667,2,8,Middle East,Western Asia
11598,YEM,Yemen (North Yemen),2024-12-01,61.166667,3,23,Middle East,Western Asia
11599,ZAF,South Africa,1989-02-01,0.000000,1,1,Africa,Sub-Saharan Africa
11600,ZAF,South Africa,1989-04-01,359.000000,1,55,Africa,Sub-Saharan Africa


# **2. Agreements**
---

## Peace agreements (PA-X) preprocessing

This section processes the PA-X agreements dataset to create country–month indicators of peace agreement activity.

1. **Load and standardize:** Column names are lowercased, the conflict identifier is harmonized (`ucdpcon → conflict_id`), and agreement dates are converted to month-level timestamps.

2. **Country expansion:** Agreements listing two parties (`loc1iso`, `loc2iso`) are reshaped so that each agreement contributes one row per involved country (ISO3).

3. **Agreement classification:** Binary indicators are created for agreement types based on `stage` and `stagesub` (e.g., comprehensive, substantive, ceasefire-related).

4. **Collapse to country–month:** Data are aggregated to the country–month level, producing:
   - dummies for whether at least one agreement of each type occurred,
   - counts of agreements by type,
   - and additional agreement-content indicators (heterogeneity features) using monthly maxima.



In [5]:
df_pax = pd.read_csv('../../data/input/pax/pax_data_2144_agreements_v9_10.csv', low_memory=False)
df_pax.columns = df_pax.columns.str.lower()
df_pax = df_pax.rename(columns={"ucdpcon": "conflict_id"})

df_pax["dat"] = pd.to_datetime(df_pax["dat"])
df_pax["year_mo"] = df_pax["dat"].dt.to_period("M").dt.to_timestamp()

#for agreements that involve two countries, stack them into separate rows
df_pax = (
    df_pax
    .set_index([c for c in df_pax.columns if c not in ['loc1iso', 'loc2iso']])
    [['loc1iso', 'loc2iso']]
    .stack()
    .reset_index()
    .drop(columns='level_278')  # removes column indicating loc1/loc2
    .rename(columns={0: 'isocode'})
    
)

# Replace SSD -> SDN before July 2011
mask = (df_pax["isocode"] == "SSD") & (
    (df_pax["dat"].dt.year < 2011) |
    ((df_pax["dat"].dt.year == 2011) & (df_pax["dat"].dt.month <= 6))
)
df_pax.loc[mask, "isocode"] = "SDN"

df_pax = df_pax.dropna(subset=['conflict_id'])
df_pax["conflict_id"] = df_pax["conflict_id"].astype(int)


df_pax["agreement"] = 1
df_pax["subs_agreement"] = (df_pax["stage"].str.lower() == "subpar").astype(int)
df_pax["comp_agreement"] = (df_pax["stage"].str.lower() == "subcomp").astype(int)
df_pax["cea_agreement"] = (df_pax["stage"].str.lower() == "cea").astype(int)
df_pax["cea_ceamix_agreement"] = (df_pax["stagesub"].str.lower() == "ceamix").astype(int)
df_pax["cea_ceas_agreement"] = (df_pax["stagesub"].str.lower() == "ceas").astype(int)
df_pax["cea_rel_agreement"] = (df_pax["stagesub"].str.lower() == "rel").astype(int)

#define variables to test clusterd Heterogenous TE
features_cluster_columns = ['hriimon', 'ime', 'ddrprog', 'imun', 'tjrep', 'ppsaut', 'tjmis', 
                            'protgrp', 'ppsvet', 'hriibod', 'tpsloc', 'terps', 'pol', 'epsres', 
                            'impk', 'epsoth', 'polnewtemp', 'protciv', 'hrtrinc', 'ssrddr', 
                            'ceprov', 'ppsint', 'ddrdemil', 'polpar', 'polps', 'hrbor', 'hrdem',
                            'medlog', 'tjvet', 'medsubs', 'epsfis', 'tpssub', 'tpsoth', 'eleccomm', 
                            'eps', 'ssrpol', 'medgov', 'imref', 'tpsaut', 'med', 'hrii', 'cegen', 'ssrgua',
                            'ssrarm', 'tjcou', 'ppsoro', 'tjamban', 'tjvic', 'tjmech', 'polnewind', 'mpsme', 
                            'mpsjt', 'imoth', 'ppsothpr', 'ppsex', 'hrmob', 'ele', 'hrni', 'civso', 'pubad', 
                            'juscr', 'mps', 'prot', 'mpspro']
# -----------------------------
# 4) Collapse to country-month 
# -----------------------------

df_agreements = (
    df_pax
    .groupby(["isocode", "year_mo"], as_index=False)
    .agg(
        agreement=("agreement", "max"),  # at least one agreement
        comp_agreement=("comp_agreement", "max"),  
        subs_agreement=("subs_agreement", "max"),
        cea_agreement = ('cea_agreement', "max"), 
        cea_ceamix_agreement = ('cea_ceamix_agreement', "max"),
        cea_ceas_agreement = ('cea_ceas_agreement', "max"),
        cea_rel_agreement = ('cea_rel_agreement', "max"),
        ce=('ce','max'),
        agreement_count=("agtid", "count"),
        comp_agreement_count=("comp_agreement", "sum"),
        subs_agreement_count=("subs_agreement", "sum"),
        cea_agreement_count = ('cea_agreement', "sum"),
        cea_ceamix_agreement_count = ('cea_ceamix_agreement', "sum"),
        cea_ceas_agreement_count = ('cea_ceas_agreement', "sum"),
        cea_rel_agreement_count = ('cea_rel_agreement', "sum"),
        ce_count = ('ce','count'),
        **{var: (var, "max") for var in features_cluster_columns}
    )
)
df_agreements

,isocode,year_mo,agreement,comp_agreement,subs_agreement,cea_agreement,cea_ceamix_agreement,cea_ceas_agreement,cea_rel_agreement,ce,...,ppsex,hrmob,ele,hrni,civso,pubad,juscr,mps,prot,mpspro
0,AFG,1992-04-01,1,0,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
1,AFG,1993-03-01,1,0,1,0,0,0,0,2,...,1,1,2,0,0,0,0,2,0,1
2,AFG,1999-07-01,1,0,0,0,0,0,0,1,...,1,1,0,0,0,0,0,0,0,0
3,AFG,2001-12-01,1,0,1,0,0,0,0,0,...,1,0,1,2,1,1,1,2,0,0
4,AFG,2002-01-01,1,0,0,0,0,0,0,0,...,0,0,0,0,1,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1408,ZAF,1992-09-01,1,0,0,0,0,0,0,0,...,0,0,1,0,0,0,1,0,0,0
1409,ZAF,1992-11-01,1,0,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
1410,ZAF,1993-11-01,1,1,0,0,0,0,0,0,...,0,0,3,3,0,3,0,0,3,0
1411,ZAF,1994-02-01,1,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# **3. Instrumental Variables**
---


1. **Past peace agreements, excluding own**
    - <code>agree_region_out_p{chosen_range}_wdist</code>, 
    - <code>agree_subreg_out_p{chosen_range}_wdist</code> 
    - <code>agree_region_out_p{chosen_range}_wtrade</code>,
    - <code>agree_subreg_out_p{chosen_range}_wtrade</code>
    - <code>agree_region_out_p{chosen_range}_wfatal</code>
    - <code>agree_subreg_out_p{chosen_range}_wfatal</code>
    - <code>agree_region_out_p{chosen_range}_wfatal_dummy</code>
    - <code>agree_subreg_out_p{chosen_range}_wfatal_dummy</code>
    - <code>share_agree_region_p{chosen_range}_wfatal</code>
    - <code>share_agree_subreg_p{chosen_range}_wfatal</code>


    1. We create weights based on three dimensions:
        1. Distance 
        2. Volume trades
    2. Create the instrumental variables:
    3. UN voting similiraty wighted by influence veto and gdp.


2. **Influence scores in UN voting:**

    - <code>influence_veto_past_12m</code>, 
    - <code>influence_gdp_past_12m</code>, 
    - <code>influence_veto_past_12 *_friends</code> (lags of fatalities * influence_veto_past_12m)





## **3.1. Regional Peace Agreement Spillovers**

---

- Explain breafly the steps taken to create the first set of instruments.

### **Distance**
---

We create a dataframe at country level with the following variables:

- <code>dist_region_c</code> Average distance to other countries in the same region (in km)
- <code>dist_subregion_c</code> Average distance to other countries in the same subregion (in km)
- <code>inv_dist_region_c</code> Inverse of average distance to other countries in the same region in 1000 km
- <code>inv_dist_subregion_c</code> Inverse of average distance to other countries in the same subregion in 1000 km



In [6]:
df_distance = pd.read_excel('../../data/input/distance/dist_cepii.xls')

iso_fix = {
    "ANT": "CUW", 
    "PAL": "PSE",
    "ROM": "ROU",
    "TMP": "TLS",
    "ZAR": "COD",
}

df_distance["iso_o"] = df_distance["iso_o"].replace(iso_fix)
df_distance["iso_d"] = df_distance["iso_d"].replace(iso_fix)

# 1. Add South Sudan (SSD) by duplicating Sudan (SDN) dyads in pre-independence data
#---------------------------------------------------------------
# rows with SDN on at least one side
m = df_distance[["iso_o", "iso_d"]].eq("SDN").any(axis=1)

# duplicate and convert SDN -> SSD
df_ssd = df_distance.loc[m].replace({"SDN": "SSD"})

# genertes combination of SDN and SSD
df_mix = (
    df_distance.loc[(df_distance["iso_o"]=="SDN") & (df_distance["iso_d"]=="SDN")]
    .assign(iso_d="SSD")
    
)

df_mix_2 = (
    df_distance.loc[(df_distance["iso_o"]=="SDN") & (df_distance["iso_d"]=="SDN")]
    .assign(iso_o="SSD")    
)

# merge everything and drop duplicates
df_distance = (
    pd.concat([df_distance, df_ssd, df_mix, df_mix_2], ignore_index=True)
    .drop_duplicates(subset=["iso_o", "iso_d"])
    .reset_index(drop=True)
)

# 2. Add region and sub_region for isocode origin and destination
#---------------------------------------------------------------
isocodes_unique = isocodes[['alpha_3', 'region', 'sub_region']].dropna().drop_duplicates(subset=['alpha_3']).rename(columns={'alpha_3':'isocode'})
df_distance = df_distance.merge(isocodes_unique, left_on = 'iso_o', right_on = 'isocode', how = 'left').drop(columns = ['isocode']).rename(columns = {'region':'region_o', 'sub_region':'sub_region_o'})
df_distance = df_distance.merge(isocodes_unique, left_on = 'iso_d', right_on = 'isocode', how = 'left').drop(columns = ['isocode']).rename(columns = {'region':'region_d', 'sub_region':'sub_region_d'})
df_distance = df_distance.dropna(subset=['region_o', 'region_d', 'sub_region_o', 'sub_region_d']).reset_index(drop=True)

# 3. Create conditional distance variables
#---------------------------------------------------------------
df_distance["dist_region_c"] = df_distance["dist"].where(df_distance["region_o"].eq(df_distance["region_d"]))
df_distance["dist_subregion_c"] = df_distance["dist"].where(df_distance["sub_region_o"].eq(df_distance["sub_region_d"]))

df_distance = df_distance.rename(columns={"iso_o": "isocode"})  

# 4. Aggregate to country level
#---------------------------------------------------------------
df_distance = (
    df_distance.groupby("isocode", as_index=False)[["dist_region_c", "dist_subregion_c"]]
      .mean()
)
df_distance['dist_region_c'] = df_distance['dist_region_c']/1000
df_distance['dist_subregion_c'] = df_distance['dist_subregion_c']/1000
df_distance['inv_dist_region_c'] = 1 / df_distance['dist_region_c']
df_distance['inv_dist_subregion_c'] = 1 / df_distance['dist_subregion_c']
df_distance

,isocode,dist_region_c,dist_subregion_c,inv_dist_region_c,inv_dist_subregion_c
0,ABW,2.071563,1.880935,0.482727,0.531651
1,AFG,3.126898,1.795700,0.319806,0.556886
2,AGO,3.004178,2.760024,0.332870,0.362316
3,AIA,2.169005,2.056058,0.461041,0.486368
4,ALB,1.365408,0.965066,0.732382,1.036199
...,...,...,...,...,...
220,YEM,4.480092,1.979152,0.223210,0.505267
221,YUG,1.190139,0.702986,0.840238,1.422504
222,ZAF,4.612814,4.219738,0.216787,0.236982
223,ZMB,3.393751,3.071870,0.294659,0.325535


### **Volume traded**
---
We create a dataframe at country level with the following variables:

- <code>value_export_region_c</code> Value of total (95 to 23) exports to the country's region in mil USD
- <code>value_export_subregion_c</code> Value of total (95 to 23) exports to the country's subregion in mil USD
- <code>total_export_world</code> Value of total (95 to 23) exports of the country in mil USD
- <code>percent_export_region_c</code> Percent of country's exports that go to its region
- <code>percent_export_subregion_c</code> Percent of country's exports that go to its subregion

In [7]:
# 1. Read + stack yearly BACI files (keep only columns we need)
#---------------------------------------------------------------
trade_dir =   "../../data/input/trade/BACI_HS92_V202501"
years = range(1995, 2024)
dfs = []
for y in years:
    f = trade_dir + f"/BACI_HS92_Y{y}_V202501.csv"
    tmp = pd.read_csv(f, usecols=["t", "i", "j", "v"])
    tmp = tmp.rename(columns={"t": "year", "i": "loc1iso", "j": "loc2iso", "v": "value_export"})
    dfs.append(tmp)

trade = pd.concat(dfs, ignore_index=True)

# 2. Collapse to exporter-importer-year totals
#---------------------------------------------------------------
trade = (
    trade.groupby(["year", "loc1iso", "loc2iso"], as_index=False)["value_export"]
         .sum()
)

# 3. Map numeric country codes to ISO3
#---------------------------------------------------------------
country_codes = dict(pd.read_csv("../../data/input/trade/BACI_HS92_V202501/country_codes_V202501.csv", usecols = ['country_code', 'country_iso3']).values)
trade['loc1iso'] = trade['loc1iso'].map(country_codes)
trade['loc2iso'] = trade['loc2iso'].map(country_codes)

# 4. Fix legacy ISO codes to modern equivalents
#---------------------------------------------------------------
iso_fix = {
    "ANT": "CUW", 
    "PAL": "PSE",
    "ROM": "ROU",
    "TMP": "TLS",
    "ZAR": "COD",
}

trade["loc1iso"] = trade["loc1iso"].replace(iso_fix)
trade["loc2iso"] = trade["loc2iso"].replace(iso_fix)

# 5. Attach region/subregion to both exporter and importer
#---------------------------------------------------------------
trade = trade.merge(isocodes_unique, left_on = 'loc1iso', right_on = 'isocode', how = 'left').drop(columns = ['isocode']).rename(columns = {'region':'region_1', 'sub_region':'sub_region_1'})
trade = trade.merge(isocodes_unique, left_on = 'loc2iso', right_on = 'isocode', how = 'left').drop(columns = ['isocode']).rename(columns = {'region':'region_2', 'sub_region':'sub_region_2'})

# 6. Keep export values only for same-region / same-subregion pairs
#---------------------------------------------------------------
trade["value_export_region_c"] = trade["value_export"].where(trade["region_1"].eq(trade["region_2"]))
trade["value_export_subregion_c"] = trade["value_export"].where(trade["sub_region_1"].eq(trade["sub_region_2"]))
trade = trade.rename(columns={"loc1iso": "isocode"})  

# 7. Aggregate to country level (exporter only)
#---------------------------------------------------------------
trade = (
    trade.groupby("isocode", as_index=False)
         .agg(
             value_export_region_c=("value_export_region_c", "sum"),
             value_export_subregion_c=("value_export_subregion_c", "sum"),
             total_export_world=("value_export", "sum")
         )
)

# 8. Compute trade shares and scale to million USD
#---------------------------------------------------------------
trade["percent_export_region_c"] = trade["value_export_region_c"] / trade["total_export_world"]
trade["percent_export_subregion_c"] = trade["value_export_subregion_c"] / trade["total_export_world"]
# --- thousands USD -> million USD (divide by 1000)
cols_musd = ["value_export_region_c", "value_export_subregion_c", "total_export_world"]
trade[cols_musd] = trade[cols_musd] / 1000

trade

,isocode,value_export_region_c,value_export_subregion_c,total_export_world,percent_export_region_c,percent_export_subregion_c
0,ABW,36183.935110,8704.570708,4.369066e+04,0.828185,0.199232
1,AFG,18072.623320,13595.311030,2.163794e+04,0.835229,0.628309
2,AGO,41010.808859,40714.715138,9.505019e+05,0.043146,0.042835
3,AIA,279.302181,190.726840,4.544570e+02,0.614584,0.419681
4,ALB,41993.301702,33244.056255,5.228000e+04,0.803238,0.635885
...,...,...,...,...,...,...
225,YEM,111308.138361,13625.641467,1.304372e+05,0.853346,0.104461
226,ZA1,0.000000,0.000000,1.579377e+05,0.000000,0.000000
227,ZAF,529021.979739,514758.821309,2.411610e+06,0.219365,0.213450
228,ZMB,39626.286120,34946.354908,2.153975e+05,0.183968,0.162241


### **Merge Distance and Volume traded**

In [8]:
# 5) Merge metadata (region/subregion, distance, trade)
df_agreements = (
    df_agreements
    .merge(isocodes_unique, on="isocode", how="left")
    .merge(df_distance[['isocode','inv_dist_region_c', 'inv_dist_subregion_c']], on="isocode", how="left")
    .merge(trade[['isocode','percent_export_region_c', 'percent_export_subregion_c']], on="isocode", how="left")
)

In [9]:
# 6) Totals per country / region / subregion and excl variables
# -----------------------------
keys_reg = ["region", "year_mo"]
keys_sub = ["sub_region", "year_mo"]

df_agreements["agreement_region"] = df_agreements.groupby(keys_reg)["agreement_count"].transform("sum")
df_agreements["agreement_subregion"] = df_agreements.groupby(keys_sub)["agreement_count"].transform("sum")

df_agreements["agreement_region_excl"] = df_agreements["agreement_region"] - df_agreements["agreement_count"]
df_agreements["agreement_subregion_excl"] = df_agreements["agreement_subregion"] - df_agreements["agreement_count"]

# -----------------------------
# 7) Fill missing weights with global mean
# -----------------------------
weight_cols = [
    "percent_export_region_c", "percent_export_subregion_c",
    "inv_dist_region_c", "inv_dist_subregion_c"
]
for c in weight_cols:
    df_agreements[c] = df_agreements[c].fillna(df_agreements[c].mean()) #TODO fill by mean of isocode's region/subregion

df_agreements['agree_region_excl_w_trade'] = df_agreements['agreement_region_excl'] * df_agreements['percent_export_region_c']
df_agreements['agree_subreg_excl_w_trade'] = df_agreements['agreement_subregion_excl'] * df_agreements['percent_export_subregion_c']
df_agreements['agree_region_excl_w_dist'] = df_agreements['agreement_region_excl'] * df_agreements['inv_dist_region_c']
df_agreements['agree_subreg_excl_w_dist'] = df_agreements['agreement_subregion_excl'] * df_agreements['inv_dist_subregion_c']

df_agreements = df_agreements.drop(columns=weight_cols + ['agreement_region', 'agreement_subregion', 'agreement_region_excl', 'agreement_subregion_excl'])
df_agreements

,isocode,year_mo,agreement,comp_agreement,subs_agreement,cea_agreement,cea_ceamix_agreement,cea_ceas_agreement,cea_rel_agreement,ce,...,juscr,mps,prot,mpspro,region,sub_region,agree_region_excl_w_trade,agree_subreg_excl_w_trade,agree_region_excl_w_dist,agree_subreg_excl_w_dist
0,AFG,1992-04-01,1,0,1,0,0,0,0,0,...,0,0,0,0,Asia,Southern Asia,0.000000,0.000000,0.000000,0.000000
1,AFG,1993-03-01,1,0,1,0,0,0,0,2,...,0,2,0,1,Asia,Southern Asia,0.000000,0.000000,0.000000,0.000000
2,AFG,1999-07-01,1,0,0,0,0,0,0,1,...,0,0,0,0,Asia,Southern Asia,0.000000,0.000000,0.000000,0.000000
3,AFG,2001-12-01,1,0,1,0,0,0,0,0,...,1,2,0,0,Asia,Southern Asia,0.000000,0.000000,0.000000,0.000000
4,AFG,2002-01-01,1,0,0,0,0,0,0,0,...,0,0,0,0,Asia,Southern Asia,0.835229,0.000000,0.319806,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1408,ZAF,1992-09-01,1,0,0,0,0,0,0,0,...,1,0,0,0,Africa,Sub-Saharan Africa,0.000000,0.000000,0.000000,0.000000
1409,ZAF,1992-11-01,1,0,1,0,0,0,0,0,...,0,0,0,0,Africa,Sub-Saharan Africa,0.219365,0.213450,0.216787,0.236982
1410,ZAF,1993-11-01,1,1,0,0,0,0,0,0,...,0,0,3,0,Africa,Sub-Saharan Africa,0.000000,0.000000,0.000000,0.000000
1411,ZAF,1994-02-01,1,0,1,0,0,0,0,0,...,0,0,0,0,Africa,Sub-Saharan Africa,0.438729,0.426901,0.433575,0.473963


## **3.2. Security Council - Voting Similarity**
---

- Members of UN
- Voting Data to generate a voting similarity score

In [10]:
members_sc = pd.read_csv('../../data/input/un/DPPA-SCMEMBERSHIP.csv', usecols=['year', 'security_council_member']).rename(columns={'security_council_member':'country'})
members_sc = members_sc[members_sc['year'] >= 1989].copy()
members_sc.loc[members_sc['country'].isna() & members_sc['year'].isin([1990, 1991, 2018, 2019]), 'country'] = "Ivory Coast"
members_sc.loc[members_sc['country'].isna() & members_sc['year'].isin([2009, 2010]), 'country'] = "Turkey"
members_sc = members_sc.merge(isocodes[['name', 'alpha_3']], left_on='country', right_on='name', how='left').rename(columns = {'alpha_3':'isocode'}).drop(columns=['name'])
members_sc

,year,country,isocode
0,1996,Indonesia,IDN
1,1996,Italy,ITA
2,1996,Poland,POL
3,1996,Republic of Korea,KOR
4,1996,Russian Federation,RUS
...,...,...,...
565,2026,United Kingdom of Great Britain and Northern I...,GBR
566,2026,Russian Federation,RUS
567,2026,France,FRA
568,2026,China,CHN


In [11]:
votes = pd.read_csv('../../data/input/un/2025_9_19_ga_voting.csv', low_memory=False, usecols=['ms_code', 'ms_name', 'ms_vote', 'date', 'resolution'])
#convert date to datetime
votes['date'] = pd.to_datetime(votes['date'])
votes = votes.loc[votes['date'].dt.year>=1968].copy()
votes['year'] = votes['date'].dt.year

#keep only rows where ms_vote is Y, N, A
votes = votes.loc[votes['ms_vote'].isin(['Y', 'N', 'A'])].copy().rename(columns={'ms_vote':'vote'})
vote_map = {'Y': 1, 'N': 0, 'A': 2}
votes['vote'] = votes['vote'].map(vote_map)

#fix some isocodes
code_fix = {
    "GER": "DEU",
    "CSK": "CZE",
    "SUN": "RUS"
}
votes['ms_code'] = votes['ms_code'].replace(code_fix)
votes = votes.merge(isocodes[['alpha_3', 'name']], left_on='ms_code', right_on='alpha_3', how='left').rename(columns = {'name':'country', 'alpha_3':'isocode'})
#capitalize the values in country column
votes['country'] = votes['country'].str.title()

#keep only these columns isocode ms_vote date, year resolution
votes = votes[['isocode','vote', 'date', 'year', 'resolution']]
votes

,isocode,vote,date,year,resolution
0,AFG,1,1985-11-01,1985,A/RES/40/6
1,AFG,1,1985-11-01,1985,A/RES/40/6
2,ALB,1,1985-11-01,1985,A/RES/40/6
3,ALB,1,1985-11-01,1985,A/RES/40/6
4,DZA,1,1985-11-01,1985,A/RES/40/6
...,...,...,...,...,...
1818262,VNM,1,2025-08-26,2025,A/RES/79/326
1818263,YEM,1,2025-08-26,2025,A/RES/79/326
1818264,YEM,1,2025-08-26,2025,A/RES/79/326
1818265,YEM,1,2025-08-26,2025,A/RES/79/326


- How aligned was country i with country j, according to their voting history in the UNGA (window y-20 to y-5).

In [12]:
from itertools import combinations

def compute_similarity(votes_window: pd.DataFrame) -> pd.DataFrame:
    """
    votes_window columns:
    - resolution
    - isocode
    - vote
    """

    pairs = []

    # Iteramos resolución por resolución
    for _, g in votes_window.groupby('resolution'):
        # Diccionario país -> voto
        d = dict(zip(g['isocode'], g['vote']))

        # Comparar todos los pares dentro de la resolución
        for c1, c2 in combinations(d.keys(), 2):
            pairs.append({
                'isocode1': c1,
                'isocode2': c2,
                'same_vote': int(d[c1] == d[c2])
            })

    if not pairs:
        return pd.DataFrame(columns=['isocode1', 'isocode2', 'same_vote'])

    pairs_df = pd.DataFrame(pairs)

    # Promedio por par de países
    similarity = (
        pairs_df
        .groupby(['isocode1', 'isocode2'], as_index=False)
        ['same_vote']
        .mean()
    )

    return similarity

sim_by_year = {}

for y in range(1989, 2024):
    print(f"Computing similarity for year {y}...")
    lo, hi = y - 20, y - 5

    votes_window = votes.loc[
        (votes['year'] >= lo) &
        (votes['year'] <= hi),
        ['resolution', 'isocode', 'vote']
    ]

    if votes_window.empty:
        sim_by_year[y] = pd.DataFrame(
            columns=['isocode1', 'isocode2', 'same_vote']
        )
        continue

    sim_by_year[y] = compute_similarity(votes_window)

Computing similarity for year 1989...
Computing similarity for year 1990...
Computing similarity for year 1991...
Computing similarity for year 1992...
Computing similarity for year 1993...
Computing similarity for year 1994...
Computing similarity for year 1995...
Computing similarity for year 1996...
Computing similarity for year 1997...
Computing similarity for year 1998...
Computing similarity for year 1999...
Computing similarity for year 2000...
Computing similarity for year 2001...
Computing similarity for year 2002...
Computing similarity for year 2003...
Computing similarity for year 2004...
Computing similarity for year 2005...
Computing similarity for year 2006...
Computing similarity for year 2007...
Computing similarity for year 2008...
Computing similarity for year 2009...
Computing similarity for year 2010...
Computing similarity for year 2011...
Computing similarity for year 2012...
Computing similarity for year 2013...
Computing similarity for year 2014...
Computing si

#### **Preprocess GDP**

- gdp_normalized
- log_gdp_normalized

Then they will be weighted by influence scores.


In [13]:
gdp = pd.read_csv('../../data/input/gdp_pc/gdp_pc.csv', low_memory=False).rename(columns={'gdp_current_usd':'gdp'})
#keep only the dates for each isocode in the month of June (real annual indicator)
gdp['year_mo'] = pd.to_datetime(gdp['year_mo'])
gdp = gdp.loc[gdp['year_mo'].dt.month==6].copy()
gdp['year'] = gdp['year_mo'].dt.year
gdp = gdp.loc[gdp['year']>=1989][['isocode', 'year', 'gdp']].copy()

# counts missing GDP (annual) per isocode and drops countries with missing >= 20.
missing_by_iso = (
    gdp.assign(missing_gdp=lambda x: x['gdp'].isna())
         .groupby('isocode', as_index=False)['missing_gdp']
         .sum()
)

valid_isos = set(missing_by_iso.loc[missing_by_iso['missing_gdp'] < 20, 'isocode'])
gdp = gdp[gdp['isocode'].isin(valid_isos)].copy()

# -----------------------------
# FORWARD FILL THEN BACKWARD FILL (ANNUAL)
# -----------------------------
gdp = gdp.sort_values(['isocode', 'year'])
gdp['gdp'] = gdp.groupby('isocode')['gdp'].ffill().bfill()


gdp['log_gdp'] = np.log1p(gdp['gdp'])
gdp['max_gdp'] = gdp.groupby('year')['gdp'].transform('max')
gdp['max_log_gdp'] = gdp.groupby('year')['log_gdp'].transform('max')
gdp['gdp_normalized'] = gdp['gdp'] / gdp['max_gdp']
gdp['log_gdp_normalized'] = gdp['log_gdp'] / gdp['max_log_gdp']

gdp = gdp.drop(columns=['max_gdp', 'max_log_gdp'])
gdp

,isocode,year,gdp,log_gdp,gdp_normalized,log_gdp_normalized
0,ABW,1989,6.955307e+08,20.360186,0.000034,0.664359
12,ABW,1990,7.648045e+08,20.455131,0.000033,0.664923
24,ABW,1991,8.720670e+08,20.586377,0.000036,0.668264
36,ABW,1992,9.586592e+08,20.681046,0.000038,0.669907
48,ABW,1993,1.083240e+09,20.803223,0.000042,0.673502
...,...,...,...,...,...,...
109832,ZWE,2020,3.198033e+10,24.188387,0.000372,0.753900
109844,ZWE,2021,4.128767e+10,24.443830,0.000421,0.758715
109856,ZWE,2022,4.075756e+10,24.430907,0.000399,0.757349
109868,ZWE,2023,3.587178e+10,24.303217,0.000336,0.752388


In [14]:
# -----------------------------
# 2) BUILD INFLUENCE SCORES FROM sim_by_year
# -----------------------------
# Interpretation:
# - isocode1: the "recipient" country i (instrument will be assigned to it)
# - isocode2: the "influencer" country j (Security Council members for year y)
# influence_{i,y} = sum_{j in SC(y)} same_vote_{i,j,y} * weight_{j,y}

# P5 set for the veto factor
p5 = {'USA', 'RUS', 'FRA', 'CHN', 'GBR'}

influence_scores_list = []

for y in range(1989, 2024):
    sim_y = sim_by_year.get(y)

    # Skip if similarity for this year is missing/empty
    if sim_y is None or sim_y.empty:
        continue

    # Get Security Council members in year y (these are isocode2)
    members_y = (
        members_sc.loc[members_sc['year'] == y, 'isocode']
                  .dropna()
                  .unique()
    )
    if len(members_y) == 0:
        continue

    # Keep only pairs where isocode2 is a SC member in year y
    sim_y = sim_y[sim_y['isocode2'].isin(members_y)].copy()
    if sim_y.empty:
        continue

    # Bring GDP weights for the SC members (isocode2) in year y
    gdp_weights_y = gdp.loc[gdp['year'] == y, ['isocode', 'gdp_normalized', 'log_gdp_normalized']].copy()
    gdp_weights_y = gdp_weights_y.rename(columns={'isocode': 'isocode2'})

    sim_y = sim_y.merge(gdp_weights_y, on='isocode2', how='left')

    # Veto factor (P5 = 10, others = 1), matches your Stata logic
    sim_y['veto_factor'] = np.where(sim_y['isocode2'].isin(p5), 10.0, 1.0)

    # Multiply weights by same_vote (this matches: replace var = var * same_vote)
    sim_y['inf_raw'] = sim_y['same_vote']
    sim_y['inf_gdp'] = sim_y['same_vote'] * sim_y['gdp_normalized']
    sim_y['inf_log_gdp'] = sim_y['same_vote'] * sim_y['log_gdp_normalized']
    sim_y['inf_veto'] = sim_y['same_vote'] * sim_y['veto_factor']

    # If GDP is missing for some SC member in year y, treat its GDP contribution as 0
    # (This is conservative; alternatively you can drop those members entirely.)
    sim_y[['inf_gdp', 'inf_log_gdp']] = sim_y[['inf_gdp', 'inf_log_gdp']].fillna(0.0)

    # Collapse by isocode1 (recipient country) -> create country-year influence scores
    infl_y = (
        sim_y.groupby('isocode1', as_index=False)
             .agg(
                 influence=('inf_raw', 'sum'),
                 influence_gdp=('inf_gdp', 'sum'),
                 influence_log_gdp=('inf_log_gdp', 'sum'),
                 influence_veto=('inf_veto', 'sum')
             )
             .rename(columns={'isocode1': 'isocode'})
    )
    infl_y['year'] = y

    influence_scores_list.append(infl_y)

# Final country-year dataset of influence scores
influence_scores = pd.concat(influence_scores_list, ignore_index=True)
influence_scores

,isocode,influence,influence_gdp,influence_log_gdp,influence_veto,year
0,AFG,10.252666,0.158361,7.657527,32.183798,1989
1,AGO,9.484779,0.128503,6.975217,30.821532,1989
2,ALB,9.943057,0.127007,7.399458,31.664799,1989
3,ARE,2.202373,0.098517,1.126583,13.469336,1989
4,ARG,9.641170,0.176694,7.196914,31.990897,1989
...,...,...,...,...,...,...
6166,UGA,1.425625,0.050265,1.230905,6.358985,2023
6167,UKR,1.697464,0.115014,1.504559,11.697269,2023
6168,VCT,2.162606,0.060698,1.862162,7.793141,2023
6169,WSM,2.297924,0.080014,1.990968,9.456259,2023


In [15]:
df_agreements['year'] = df_agreements['year_mo'].dt.year
df_agreements = df_agreements.merge(influence_scores, on=['isocode', 'year'], how='left')
#fill missing influence scores with 0
influence_cols = ['influence', 'influence_gdp', 'influence_log_gdp', 'influence_veto']
for c in influence_cols:
    df_agreements[c] = df_agreements[c].fillna(0)
df_agreements

,isocode,year_mo,agreement,comp_agreement,subs_agreement,cea_agreement,cea_ceamix_agreement,cea_ceas_agreement,cea_rel_agreement,ce,...,sub_region,agree_region_excl_w_trade,agree_subreg_excl_w_trade,agree_region_excl_w_dist,agree_subreg_excl_w_dist,year,influence,influence_gdp,influence_log_gdp,influence_veto
0,AFG,1992-04-01,1,0,1,0,0,0,0,0,...,Southern Asia,0.000000,0.000000,0.000000,0.000000,1992,9.798499,0.193068,7.936617,31.424997
1,AFG,1993-03-01,1,0,1,0,0,0,0,2,...,Southern Asia,0.000000,0.000000,0.000000,0.000000,1993,9.881162,0.198768,7.957132,31.361555
2,AFG,1999-07-01,1,0,0,0,0,0,0,1,...,Southern Asia,0.000000,0.000000,0.000000,0.000000,1999,9.722973,0.127616,7.753562,31.064270
3,AFG,2001-12-01,1,0,1,0,0,0,0,0,...,Southern Asia,0.000000,0.000000,0.000000,0.000000,2001,10.393686,0.113194,8.296902,32.021750
4,AFG,2002-01-01,1,0,0,0,0,0,0,0,...,Southern Asia,0.835229,0.000000,0.319806,0.000000,2002,10.483315,0.140090,8.439069,32.423170
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1408,ZAF,1992-09-01,1,0,0,0,0,0,0,0,...,Sub-Saharan Africa,0.000000,0.000000,0.000000,0.000000,1992,1.933565,0.208948,1.740907,16.614715
1409,ZAF,1992-11-01,1,0,1,0,0,0,0,0,...,Sub-Saharan Africa,0.219365,0.213450,0.216787,0.236982,1992,1.933565,0.208948,1.740907,16.614715
1410,ZAF,1993-11-01,1,1,0,0,0,0,0,0,...,Sub-Saharan Africa,0.000000,0.000000,0.000000,0.000000,1993,2.286180,0.223810,2.051345,16.919834
1411,ZAF,1994-02-01,1,0,1,0,0,0,0,0,...,Sub-Saharan Africa,0.438729,0.426901,0.433575,0.473963,1994,1.666667,0.207257,1.526133,13.666667


## **3.3. External Suport**
---

- Explain breafly the steps taken to create the external support variable.

In [16]:
esd = pd.read_excel('../../data/input/external_support/ucdp-esd-ty-181.xlsx', usecols=['id', 'year', 'ext_name','ext_sup', 'ext_nonstate', 'active', 'country_a'])

# 1) Filter: state actors, active, year >= 1989
esd = esd.loc[
    (esd['active'] == 1) & #active dyad year
    (esd['year'] >= 1989) &
    (esd['ext_sup'] == 1) & #external support was provided
    (esd['ext_nonstate'] == 0) #external supporter is a state
].copy()
esd

# 2) Clean ext_name: remove "Government of "
esd['ext_name'] = esd['ext_name'].str.replace('Government of ', '', regex=False).str.strip()
esd['country_a'] = esd['country_a'].str.strip()


# 3) Map external supporter country name -> isocodes
esd = esd.merge(
    isocodes[['name', 'alpha_3']], left_on='ext_name', right_on='name', how='left'
).rename(columns={'alpha_3': 'isocode_supporter'}).drop(columns=['name'])

isocodes_dict = {'Vietnam (North Vietnam)': 'VNM', 
                'Nepalese elements': 'NPL',
                'East Germany': 'DEU',
                'Sudanese elements': 'SDN',
                'South African elements': 'ZAF',
                'Bangladeshi elements': 'BGD', 
                'Indian elements': 'IND', 
                'Bhutanese elements': 'BTN',
                'Russian elements': 'RUS',
                'Malaysian elements': 'MYS',
                'Congolese elements': 'COG'}
esd['isocode_supporter'] = esd.apply(
    lambda row: isocodes_dict[row['ext_name']] if pd.isna(row['isocode_supporter']) and row['ext_name'] in isocodes_dict else row['isocode_supporter'],
    axis=1
)
esd = esd.dropna(subset=['isocode_supporter']).reset_index(drop=True)

# 4) Map war location country_a -> ISO3, region, sub_region
esd = esd.merge(
    isocodes[['name', 'alpha_3', 'region', 'sub_region']], left_on='country_a', right_on='name', how='left'
).rename(columns={'alpha_3': 'isocode_war_location'}).drop(columns=['name'])



# 5) Keep only supporters that are Security Council members in that year
# This creates: "SC members supporting conflicts elsewhere"
sc = members_sc.rename(columns={'isocode': 'isocode_supporter'})[['year', 'isocode_supporter']].drop_duplicates()

esd_sc = esd.merge(sc, on=['year', 'isocode_supporter'], how='inner')



# 6) Build involvement indicator by (year, isocode_supporter, isocode_war_location)
# We want a 0/1 per supporter-country per year per location:
# "did this SC member support any conflict in this country this year?"
invol = (
    esd_sc.groupby(['year', 'isocode_supporter', 'isocode_war_location', 'region', 'sub_region'], as_index=False)
          .agg(n_events=('id', 'nunique'))
)


# 8) Count how many distinct SC members were involved in each war location each year
# Sum of any_involvement across supporter_iso gives:
# "number of SC members involved in conflicts in that location that year"
loc_year = (
    invol.groupby(['year', 'isocode_war_location', 'sub_region', 'region'], as_index=False)
         .agg(sc_supporter_involved=('isocode_supporter', 'nunique'))
)


# 9) COMPUTE "OUTSIDE" MEASURES
# total SC involvement across all countries in that year
loc_year['total_involvement'] = loc_year.groupby('year')['sc_supporter_involved'].transform('sum')

# total SC involvement within the same region / sub_region
loc_year['involvement_region'] = loc_year.groupby(['year', 'region'])['sc_supporter_involved'].transform('sum')
loc_year['involvement_sub_region'] = loc_year.groupby(['year', 'sub_region'])['sc_supporter_involved'].transform('sum')

# outside definitions
loc_year['sc_at_war_outside_isocode'] = (loc_year['total_involvement'] - loc_year['sc_supporter_involved'])/loc_year['total_involvement']
loc_year['sc_at_war_outside_sub_region'] = (loc_year['total_involvement'] - loc_year['involvement_sub_region'])/loc_year['total_involvement']
loc_year['sc_at_war_outside_region'] = (loc_year['total_involvement'] - loc_year['involvement_region'])/loc_year['total_involvement']

# Final dataset keyed by conflict country ISO3 and year
external_support = (
    loc_year.rename(columns={'isocode_war_location': 'isocode'})
            [['year', 'isocode', 'sub_region', 'region',
              'sc_at_war_outside_isocode', 'sc_at_war_outside_sub_region', 'sc_at_war_outside_region']]
)
external_support

,year,isocode,sub_region,region,sc_at_war_outside_isocode,sc_at_war_outside_sub_region,sc_at_war_outside_region
0,1989,AFG,Southern Asia,Asia,0.933333,0.900000,0.633333
1,1989,AGO,Sub-Saharan Africa,Africa,0.933333,0.766667,0.600000
2,1989,ETH,Sub-Saharan Africa,Africa,0.966667,0.766667,0.600000
3,1989,GTM,Latin America and the Caribbean,Americas,0.966667,0.766667,0.766667
4,1989,IND,Southern Asia,Asia,0.966667,0.900000,0.633333
...,...,...,...,...,...,...,...
520,2017,TCD,Sub-Saharan Africa,Africa,0.961039,0.610390,0.480519
521,2017,THA,South-eastern Asia,Asia,0.987013,0.922078,0.571429
522,2017,TUR,Western Asia,Asia,0.974026,0.792208,0.571429
523,2017,UKR,Eastern Europe,Europe,0.961039,0.948052,0.948052


In [17]:
df_agreements = df_agreements.merge(external_support[['year', 'isocode', 'sc_at_war_outside_isocode', 'sc_at_war_outside_sub_region', 'sc_at_war_outside_region']], 
                                    on=['isocode', 'year'], how='left')

# 1) Country-level: if missing, default to 1
df_agreements['sc_at_war_outside_isocode'] = df_agreements['sc_at_war_outside_isocode'].fillna(1)

# 2) Subregion-level fill: use same (year, sub_region) value if available, else 1
sub_fill = (
    df_agreements.groupby(['year', 'sub_region'])['sc_at_war_outside_sub_region']
    .transform('first') 
)
df_agreements['sc_at_war_outside_sub_region'] = (
    df_agreements['sc_at_war_outside_sub_region']
    .fillna(sub_fill)
    .fillna(1)
)

# 3) Region-level fill: use same (year, region) value if available, else 1
reg_fill = (
    df_agreements.groupby(['year', 'region'])['sc_at_war_outside_region']
    .transform('first')
)
df_agreements['sc_at_war_outside_region'] = (
    df_agreements['sc_at_war_outside_region']
    .fillna(reg_fill)
    .fillna(1)
)
df_agreements

,isocode,year_mo,agreement,comp_agreement,subs_agreement,cea_agreement,cea_ceamix_agreement,cea_ceas_agreement,cea_rel_agreement,ce,...,agree_region_excl_w_dist,agree_subreg_excl_w_dist,year,influence,influence_gdp,influence_log_gdp,influence_veto,sc_at_war_outside_isocode,sc_at_war_outside_sub_region,sc_at_war_outside_region
0,AFG,1992-04-01,1,0,1,0,0,0,0,0,...,0.000000,0.000000,1992,9.798499,0.193068,7.936617,31.424997,0.958333,0.916667,0.666667
1,AFG,1993-03-01,1,0,1,0,0,0,0,2,...,0.000000,0.000000,1993,9.881162,0.198768,7.957132,31.361555,0.950000,0.900000,0.600000
2,AFG,1999-07-01,1,0,0,0,0,0,0,1,...,0.000000,0.000000,1999,9.722973,0.127616,7.753562,31.064270,0.954545,0.863636,0.590909
3,AFG,2001-12-01,1,0,1,0,0,0,0,0,...,0.000000,0.000000,2001,10.393686,0.113194,8.296902,32.021750,0.826087,0.695652,0.521739
4,AFG,2002-01-01,1,0,0,0,0,0,0,0,...,0.319806,0.000000,2002,10.483315,0.140090,8.439069,32.423170,0.944444,0.777778,0.500000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1408,ZAF,1992-09-01,1,0,0,0,0,0,0,0,...,0.000000,0.000000,1992,1.933565,0.208948,1.740907,16.614715,1.000000,0.583333,0.541667
1409,ZAF,1992-11-01,1,0,1,0,0,0,0,0,...,0.216787,0.236982,1992,1.933565,0.208948,1.740907,16.614715,1.000000,0.583333,0.541667
1410,ZAF,1993-11-01,1,1,0,0,0,0,0,0,...,0.000000,0.000000,1993,2.286180,0.223810,2.051345,16.919834,1.000000,0.700000,0.450000
1411,ZAF,1994-02-01,1,0,1,0,0,0,0,0,...,0.433575,0.473963,1994,1.666667,0.207257,1.526133,13.666667,1.000000,0.705882,0.529412


# **4. Create Panel Dataset**
---

In [40]:
df_panel = df_country_month.merge(
    df_agreements.drop(columns=['region', 'sub_region', 'year']),
    on=["isocode", "year_mo"],
    how="left"
)

df_panel

,isocode,country,year_mo,best,n_conflicts,n_events,region,subregion,agreement,comp_agreement,...,agree_subreg_excl_w_trade,agree_region_excl_w_dist,agree_subreg_excl_w_dist,influence,influence_gdp,influence_log_gdp,influence_veto,sc_at_war_outside_isocode,sc_at_war_outside_sub_region,sc_at_war_outside_region
0,AFG,Afghanistan,1989-01-01,693.750000,1,13,Asia,Southern Asia,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AFG,Afghanistan,1989-02-01,162.750000,1,18,Asia,Southern Asia,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AFG,Afghanistan,1989-03-01,1745.750000,1,21,Asia,Southern Asia,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AFG,Afghanistan,1989-04-01,495.750000,1,28,Asia,Southern Asia,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AFG,Afghanistan,1989-05-01,441.000000,1,18,Asia,Southern Asia,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11597,YEM,Yemen (North Yemen),2024-11-01,26.166667,2,8,Middle East,Western Asia,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11598,YEM,Yemen (North Yemen),2024-12-01,61.166667,3,23,Middle East,Western Asia,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11599,ZAF,South Africa,1989-02-01,0.000000,1,1,Africa,Sub-Saharan Africa,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11600,ZAF,South Africa,1989-04-01,359.000000,1,55,Africa,Sub-Saharan Africa,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# **5. Expand the panel**
---

Expand the panel from 1989-01 to 2024-12 monthly.
This is done after mergin the agreements, so we include only the interventions that occured in violent periods of the conflicts.


In [41]:
#3. Fill missing months with 0 fatalities to expand to a balanced panel
#-----------------------------------------------------------------------------------------------
# Global dataset range (for example)
global_start = df_ucdp.date_start.min()
global_end   = df_ucdp.date_end.max()

iso_to_country = df_panel.drop_duplicates('isocode').set_index('isocode')['country'].to_dict()

# crear grid de país–mes
isocodes = df_panel['isocode'].unique()
all_months = pd.date_range(global_start, global_end, freq='MS')

import itertools
full_index = pd.DataFrame(itertools.product(isocodes, all_months), columns=['isocode', 'year_mo'])

# merge con tus datos y rellenar ceros
df_panel = full_index.merge(df_panel, on=['isocode', 'year_mo'], how='left')
df_panel['country'] = df_panel['isocode'].map(iso_to_country)
df_panel['real_observation'] = np.where(df_panel['best'].notna(), 1, 0)
df_panel

,isocode,year_mo,country,best,n_conflicts,n_events,region,subregion,agreement,comp_agreement,...,agree_region_excl_w_dist,agree_subreg_excl_w_dist,influence,influence_gdp,influence_log_gdp,influence_veto,sc_at_war_outside_isocode,sc_at_war_outside_sub_region,sc_at_war_outside_region,real_observation
0,AFG,1989-01-01,Afghanistan,693.75,1.0,13.0,Asia,Southern Asia,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,AFG,1989-02-01,Afghanistan,162.75,1.0,18.0,Asia,Southern Asia,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,AFG,1989-03-01,Afghanistan,1745.75,1.0,21.0,Asia,Southern Asia,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,AFG,1989-04-01,Afghanistan,495.75,1.0,28.0,Asia,Southern Asia,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
4,AFG,1989-05-01,Afghanistan,441.00,1.0,18.0,Asia,Southern Asia,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46651,ZAF,2024-08-01,South Africa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
46652,ZAF,2024-09-01,South Africa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
46653,ZAF,2024-10-01,South Africa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
46654,ZAF,2024-11-01,South Africa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [42]:
# Fill in missing values
instruments_1 = ['agree_region_excl_w_dist', 'agree_region_excl_w_trade', 'agree_subreg_excl_w_trade', 'agree_subreg_excl_w_dist']
instruments_2 = ['influence', 'influence_gdp', 'influence_log_gdp', 'influence_veto']
instruments_3 = ['sc_at_war_outside_isocode', 'sc_at_war_outside_sub_region', 'sc_at_war_outside_region']

columns_to_fill = ['best', 'n_events', 'n_conflicts','agreement', 'comp_agreement','subs_agreement', 'cea_agreement','cea_ceas_agreement', 'cea_ceamix_agreement','cea_rel_agreement', 'ce',
                   'agreement_count','ce_count', 'cea_rel_agreement_count', 'cea_ceas_agreement_count', 'cea_ceamix_agreement_count', 'cea_agreement_count', 'subs_agreement_count', 
                   'comp_agreement_count'] + features_cluster_columns + instruments_1 + instruments_2 

for i in columns_to_fill:
    df_panel[i] = df_panel[i].fillna(0)
for i in instruments_3:
    df_panel[i] = df_panel[i].fillna(1)

df_panel['log_best'] = np.log1p(df_panel['best'])

columns_to_ffill_bfill = ['isocode', 'country', 'region', 'subregion']

for i in columns_to_ffill_bfill:
    df_panel[i] = df_panel.groupby('isocode')[i].transform(lambda x: x.ffill().bfill())


df_panel['isocode_num'] = df_panel['isocode'].astype('category').cat.codes + 1
df_panel['region_num'] = df_panel['region'].astype('category').cat.codes + 1
df_panel


,isocode,year_mo,country,best,n_conflicts,n_events,region,subregion,agreement,comp_agreement,...,influence_gdp,influence_log_gdp,influence_veto,sc_at_war_outside_isocode,sc_at_war_outside_sub_region,sc_at_war_outside_region,real_observation,log_best,isocode_num,region_num
0,AFG,1989-01-01,Afghanistan,693.75,1.0,13.0,Asia,Southern Asia,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,1.0,1,6.543552,1,3
1,AFG,1989-02-01,Afghanistan,162.75,1.0,18.0,Asia,Southern Asia,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,1.0,1,5.098341,1,3
2,AFG,1989-03-01,Afghanistan,1745.75,1.0,21.0,Asia,Southern Asia,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,1.0,1,7.465512,1,3
3,AFG,1989-04-01,Afghanistan,495.75,1.0,28.0,Asia,Southern Asia,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,1.0,1,6.208087,1,3
4,AFG,1989-05-01,Afghanistan,441.00,1.0,18.0,Asia,Southern Asia,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,1.0,1,6.091310,1,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46651,ZAF,2024-08-01,South Africa,0.00,0.0,0.0,Africa,Sub-Saharan Africa,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,1.0,0,0.000000,108,1
46652,ZAF,2024-09-01,South Africa,0.00,0.0,0.0,Africa,Sub-Saharan Africa,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,1.0,0,0.000000,108,1
46653,ZAF,2024-10-01,South Africa,0.00,0.0,0.0,Africa,Sub-Saharan Africa,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,1.0,0,0.000000,108,1
46654,ZAF,2024-11-01,South Africa,0.00,0.0,0.0,Africa,Sub-Saharan Africa,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,1.0,0,0.000000,108,1


# **6. Feature Engineering**
---

### **World Bank Indicators**
---
<code>GDP per capita</code>, <code>GDP</code> from the World Bank will be used to normalize outcome variable and as control variables.

In [43]:
gdp_pc = pd.read_csv('../../data/input/gdp_pc/gdp_pc.csv')
gdp_pc["year_mo"] = pd.to_datetime(gdp_pc["year_mo"]).dt.to_period("M").dt.to_timestamp()
df_panel = df_panel.merge(gdp_pc, left_on=["isocode", "year_mo"], right_on=["isocode", "year_mo"], how="left")
df_panel

,isocode,year_mo,country,best,n_conflicts,n_events,region,subregion,agreement,comp_agreement,...,sc_at_war_outside_isocode,sc_at_war_outside_sub_region,sc_at_war_outside_region,real_observation,log_best,isocode_num,region_num,gdp_pc_current_usd,gdp_current_usd,log_gdp_pc_current_usd
0,AFG,1989-01-01,Afghanistan,693.75,1.0,13.0,Asia,Southern Asia,0.0,0.0,...,1.0,1.0,1.0,1,6.543552,1,3,NaN,NaN,NaN
1,AFG,1989-02-01,Afghanistan,162.75,1.0,18.0,Asia,Southern Asia,0.0,0.0,...,1.0,1.0,1.0,1,5.098341,1,3,NaN,NaN,NaN
2,AFG,1989-03-01,Afghanistan,1745.75,1.0,21.0,Asia,Southern Asia,0.0,0.0,...,1.0,1.0,1.0,1,7.465512,1,3,NaN,NaN,NaN
3,AFG,1989-04-01,Afghanistan,495.75,1.0,28.0,Asia,Southern Asia,0.0,0.0,...,1.0,1.0,1.0,1,6.208087,1,3,NaN,NaN,NaN
4,AFG,1989-05-01,Afghanistan,441.00,1.0,18.0,Asia,Southern Asia,0.0,0.0,...,1.0,1.0,1.0,1,6.091310,1,3,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46651,ZAF,2024-08-01,South Africa,0.00,0.0,0.0,Africa,Sub-Saharan Africa,0.0,0.0,...,1.0,1.0,1.0,0,0.000000,108,1,NaN,NaN,NaN
46652,ZAF,2024-09-01,South Africa,0.00,0.0,0.0,Africa,Sub-Saharan Africa,0.0,0.0,...,1.0,1.0,1.0,0,0.000000,108,1,NaN,NaN,NaN
46653,ZAF,2024-10-01,South Africa,0.00,0.0,0.0,Africa,Sub-Saharan Africa,0.0,0.0,...,1.0,1.0,1.0,0,0.000000,108,1,NaN,NaN,NaN
46654,ZAF,2024-11-01,South Africa,0.00,0.0,0.0,Africa,Sub-Saharan Africa,0.0,0.0,...,1.0,1.0,1.0,0,0.000000,108,1,NaN,NaN,NaN


### **First treatment date per conflict**
---


In [44]:
df_panel['year_mo'] = pd.to_datetime(df_panel['year_mo'], format='%Y-%m').dt.to_period('M')
base_date = df_panel['year_mo'].min()
#numeric month index starting from 1
df_panel['year_mo_numeric'] = (df_panel["year_mo"] - base_date).apply(lambda x: x.n+1)

for i in ['agreement', 'comp_agreement','subs_agreement', 'cea_agreement','cea_ceas_agreement', 'cea_ceamix_agreement','cea_rel_agreement']:
    #create the group variable for first treatment date
    df_panel[f'first_{i}_date'] = (
    df_panel[df_panel[i] == 1]
    .groupby('isocode')['year_mo_numeric']
    .transform('min')
    )

    df_panel[f'first_{i}_date'] = (
    df_panel.groupby('isocode')[f'first_{i}_date']
    .transform(lambda x: x.ffill().bfill())
    )
    #Variable gvar: 0 = never treated, positive = month of the first treatment
    df_panel[f'first_{i}_date'] = df_panel[f'first_{i}_date'].fillna(0).astype(int)

    #Dummy: indicator for treated units by agreements
    df_panel[f'treated_{i}'] = (
    df_panel.groupby('isocode')[i]
    .transform('max')
    )

df_panel

,isocode,year_mo,country,best,n_conflicts,n_events,region,subregion,agreement,comp_agreement,...,first_subs_agreement_date,treated_subs_agreement,first_cea_agreement_date,treated_cea_agreement,first_cea_ceas_agreement_date,treated_cea_ceas_agreement,first_cea_ceamix_agreement_date,treated_cea_ceamix_agreement,first_cea_rel_agreement_date,treated_cea_rel_agreement
0,AFG,1989-01,Afghanistan,693.75,1.0,13.0,Asia,Southern Asia,0.0,0.0,...,40,1.0,0,0.0,0,0.0,0,0.0,0,0.0
1,AFG,1989-02,Afghanistan,162.75,1.0,18.0,Asia,Southern Asia,0.0,0.0,...,40,1.0,0,0.0,0,0.0,0,0.0,0,0.0
2,AFG,1989-03,Afghanistan,1745.75,1.0,21.0,Asia,Southern Asia,0.0,0.0,...,40,1.0,0,0.0,0,0.0,0,0.0,0,0.0
3,AFG,1989-04,Afghanistan,495.75,1.0,28.0,Asia,Southern Asia,0.0,0.0,...,40,1.0,0,0.0,0,0.0,0,0.0,0,0.0
4,AFG,1989-05,Afghanistan,441.00,1.0,18.0,Asia,Southern Asia,0.0,0.0,...,40,1.0,0,0.0,0,0.0,0,0.0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46651,ZAF,2024-08,South Africa,0.00,0.0,0.0,Africa,Sub-Saharan Africa,0.0,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
46652,ZAF,2024-09,South Africa,0.00,0.0,0.0,Africa,Sub-Saharan Africa,0.0,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
46653,ZAF,2024-10,South Africa,0.00,0.0,0.0,Africa,Sub-Saharan Africa,0.0,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
46654,ZAF,2024-11,South Africa,0.00,0.0,0.0,Africa,Sub-Saharan Africa,0.0,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0


In [45]:
df_panel.columns = df_panel.columns.str.replace(' ', '_').str.replace('-', '_').str.replace('.', '_').str.replace('|', '_')
df_panel['best'] = df_panel['best'].astype(float)
df_panel['log_best'] = np.log1p(df_panel['best'].astype(float))
df_panel

,isocode,year_mo,country,best,n_conflicts,n_events,region,subregion,agreement,comp_agreement,...,first_subs_agreement_date,treated_subs_agreement,first_cea_agreement_date,treated_cea_agreement,first_cea_ceas_agreement_date,treated_cea_ceas_agreement,first_cea_ceamix_agreement_date,treated_cea_ceamix_agreement,first_cea_rel_agreement_date,treated_cea_rel_agreement
0,AFG,1989-01,Afghanistan,693.75,1.0,13.0,Asia,Southern Asia,0.0,0.0,...,40,1.0,0,0.0,0,0.0,0,0.0,0,0.0
1,AFG,1989-02,Afghanistan,162.75,1.0,18.0,Asia,Southern Asia,0.0,0.0,...,40,1.0,0,0.0,0,0.0,0,0.0,0,0.0
2,AFG,1989-03,Afghanistan,1745.75,1.0,21.0,Asia,Southern Asia,0.0,0.0,...,40,1.0,0,0.0,0,0.0,0,0.0,0,0.0
3,AFG,1989-04,Afghanistan,495.75,1.0,28.0,Asia,Southern Asia,0.0,0.0,...,40,1.0,0,0.0,0,0.0,0,0.0,0,0.0
4,AFG,1989-05,Afghanistan,441.00,1.0,18.0,Asia,Southern Asia,0.0,0.0,...,40,1.0,0,0.0,0,0.0,0,0.0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46651,ZAF,2024-08,South Africa,0.00,0.0,0.0,Africa,Sub-Saharan Africa,0.0,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
46652,ZAF,2024-09,South Africa,0.00,0.0,0.0,Africa,Sub-Saharan Africa,0.0,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
46653,ZAF,2024-10,South Africa,0.00,0.0,0.0,Africa,Sub-Saharan Africa,0.0,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
46654,ZAF,2024-11,South Africa,0.00,0.0,0.0,Africa,Sub-Saharan Africa,0.0,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0


### **Treatment variables**
---

In [46]:
df_panel.year_mo = df_panel.year_mo.astype(str)

In [47]:
# Define Conflict Active
#================================================================================

# Define any_violence: 1 if there were fatalities, 0 otherwise
df_panel['any_violence'] = (df_panel['log_best'] > 0).astype(int)

def gen_since(series: pd.Series):
    #count the number of consecutive months since last violence
    series = (series == False)
    groups = (series == 0).cumsum()
    result = series.groupby(groups).cumsum()
    result[groups == 0] = np.nan
    return result

def gen_until(series: pd.Series) -> pd.Series:
    #count the number of consecutive months until next violence
    series = series[::-1]
    series = series == 0
    groups = (series == 0).cumsum()
    result = series.groupby(groups).cumsum()[::-1]
    result[groups == 0] = np.nan
    return result

# Apply function within each conflict panel
df_panel['since_any_violence'] = df_panel.groupby('isocode')['any_violence'].transform(gen_since)
df_panel.year_mo = df_panel.year_mo.astype(str)
df_panel['year'] = df_panel['year_mo'].str[:4].astype(int)

# Define conflict_active according to the 24-month rule (and special 12-month rule for 1990)
df_panel['conflict_active'] = np.where(
    (df_panel['since_any_violence'] < 24) |
    ((df_panel['since_any_violence'] < 12) & (df_panel['year']<= 1990)),
    1, 0
)

#create agreements associated with conflict active
# ================================================================================

for i in ['agreement', 'comp_agreement', 'subs_agreement', 'cea_agreement', 'cea_ceas_agreement', 
          'cea_rel_agreement', 'cea_ceamix_agreement', 'ce']:
    if i != 'ce':
        df_panel[f'{i}'] = np.where(
            (df_panel[f'{i}'] == 1) & (df_panel['conflict_active'] == 1),
            1, 0
        )
    else:
        df_panel[f'{i}asefire_mentions'] = np.where(
            (df_panel[f'{i}'] >= 1) & (df_panel['conflict_active'] == 1),
            1, 0)


df_panel['treated_ceasfire_mentions'] = (df_panel.groupby('isocode')['ceasefire_mentions'].transform('max'))

#Define definitive treatment variable
#--------------------------------------------------------------------------------
#combine cea_agreementes with ceasefire_mentions
df_panel['ceasfire_agreements_mentions'] = np.where(
    (df_panel['cea_agreement'] == 1) | (df_panel['ceasefire_mentions'] == 1),
    1,0
)
df_panel['treated_ceasfire_agreements_mentions'] = (df_panel.groupby('isocode')['ceasfire_agreements_mentions'].transform('max'))
df_panel

,isocode,year_mo,country,best,n_conflicts,n_events,region,subregion,agreement,comp_agreement,...,first_cea_rel_agreement_date,treated_cea_rel_agreement,any_violence,since_any_violence,year,conflict_active,ceasefire_mentions,treated_ceasfire_mentions,ceasfire_agreements_mentions,treated_ceasfire_agreements_mentions
0,AFG,1989-01,Afghanistan,693.75,1.0,13.0,Asia,Southern Asia,0,0,...,0,0.0,1,0.0,1989,1,0,1,0,1
1,AFG,1989-02,Afghanistan,162.75,1.0,18.0,Asia,Southern Asia,0,0,...,0,0.0,1,0.0,1989,1,0,1,0,1
2,AFG,1989-03,Afghanistan,1745.75,1.0,21.0,Asia,Southern Asia,0,0,...,0,0.0,1,0.0,1989,1,0,1,0,1
3,AFG,1989-04,Afghanistan,495.75,1.0,28.0,Asia,Southern Asia,0,0,...,0,0.0,1,0.0,1989,1,0,1,0,1
4,AFG,1989-05,Afghanistan,441.00,1.0,18.0,Asia,Southern Asia,0,0,...,0,0.0,1,0.0,1989,1,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46651,ZAF,2024-08,South Africa,0.00,0.0,0.0,Africa,Sub-Saharan Africa,0,0,...,0,0.0,0,423.0,2024,0,0,0,0,0
46652,ZAF,2024-09,South Africa,0.00,0.0,0.0,Africa,Sub-Saharan Africa,0,0,...,0,0.0,0,424.0,2024,0,0,0,0,0
46653,ZAF,2024-10,South Africa,0.00,0.0,0.0,Africa,Sub-Saharan Africa,0,0,...,0,0.0,0,425.0,2024,0,0,0,0,0
46654,ZAF,2024-11,South Africa,0.00,0.0,0.0,Africa,Sub-Saharan Africa,0,0,...,0,0.0,0,426.0,2024,0,0,0,0,0


### **Treatment without ceasefire mentions**

In [48]:
# months to next ceasefire mention / agreement (por conflicto)
df_panel['until_ce_mentions'] = df_panel.groupby('isocode')['ceasefire_mentions'].transform(gen_until)
df_panel['ce_mentions_active'] = (df_panel['until_ce_mentions'] >= 6).astype(int)

df_panel['until_cea_agreement'] = df_panel.groupby('isocode')['cea_agreement'].transform(gen_until)
df_panel['cea_agreement_active'] = (df_panel['until_cea_agreement'] >= 6).astype(int)

active_6m = (df_panel['ce_mentions_active'].eq(1) & df_panel['cea_agreement_active'].eq(1))


# ============================================================
# Generate no ceasefire variables for agreement, comp_agreement, subs_agreement
# ============================================================

types = ['agreement', 'comp_agreement', 'subs_agreement']

for t in types:
    base = f'{t}_no_ceasefire'
    treated = f'treated_{t}s_no_ceasefire'
    var6m = f'{t}_no_ceasefire_mentions_agreement_6m'

    # t == 1 AND ce <= 0 AND cea_agreement == 0
    df_panel[base] = (
        df_panel[t].eq(1) &
        df_panel['ce'].le(0) &
        df_panel['cea_agreement'].eq(0)
    ).astype(int)

    # max per conflict
    df_panel[treated] = df_panel.groupby('isocode')[base].transform('max')

    # (active 6m) AND base
    df_panel[var6m] = (active_6m & df_panel[base].eq(1)).astype(int)

In [49]:
df_panel.to_csv('../../data/output/country_level/country_panel.csv')